# EF_exp_gonzalo: formato de archivo, CSV frente a Parquet
**Responsable:** Alvaro Gonzalo Espinoza Vera

**Objetivo:** comparar el tamaño en disco y el tiempo de lectura más agrupación entre CSV y Parquet
para 100 000, 500 000 y 1 000 000 de filas.

**Requisito previo:** haber ejecutado `00_configuracion` y subido `sinteticos_100k.csv`, `sinteticos_500k.csv`
y `sinteticos_1m.csv` al volumen `workspace.ef_precios_medicamentos.datos`.

**Control:** misma consulta, misma cuenta y cómputo, 1 calentamiento descartado, 3 repeticiones, se reporta la mediana.

In [0]:
# 1. Configuración común del equipo (no cambiar los nombres)
CATALOGO = "workspace"
ESQUEMA = "ef_precios_medicamentos"
VOLUMEN = "datos"
BASE = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}/"
TABLA_RESULTADOS = f"{CATALOGO}.{ESQUEMA}.exp_resultados"

EXPERIMENTO = "EXP_GONZALO_FORMATO"
INTEGRANTE = "Gonzalo"
REPETICIONES = 3
GUARDAR_EN_TABLA = True   

CASOS = {"100k": "sinteticos_100k", "500k": "sinteticos_500k", "1m": "sinteticos_1m"}

print("Versión de Spark:", spark.version)
display(dbutils.fs.ls(BASE))   

Versión de Spark: 4.2.0


path,name,size,modificationTime
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_100k.csv,sinteticos_100k.csv,23993101,1795290392000
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_1m.csv,sinteticos_1m.csv,240356190,1795290407000
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_500k.csv,sinteticos_500k.csv,120131575,1795290408000


In [0]:
# 2. Verificación de los datos: conteo y nombres de columnas

COL_GRUPO_1 = "clasificacion"
COL_GRUPO_2 = "pais"
COL_VALOR = "precio_media"

for etiqueta, nombre in CASOS.items():
    df = spark.read.csv(BASE + nombre + ".csv", header=True, inferSchema=True)
    faltantes = [c for c in (COL_GRUPO_1, COL_GRUPO_2, COL_VALOR) if c.lower() not in [x.lower() for x in df.columns]]
    print(f"{etiqueta}: {df.count():,} filas | columnas faltantes: {faltantes or 'ninguna'}")

df.printSchema()

100k: 100,000 filas | columnas faltantes: ninguna
500k: 500,000 filas | columnas faltantes: ninguna
1m: 1,000,000 filas | columnas faltantes: ninguna
root
 |-- codigo_producto: double (nullable = true)
 |-- nombre_comercial: string (nullable = true)
 |-- concentracion: string (nullable = true)
 |-- forma_farmaceutica: string (nullable = true)
 |-- presentacion: string (nullable = true)
 |-- registro_sanitario: string (nullable = true)
 |-- ifa: string (nullable = true)
 |-- titular: string (nullable = true)
 |-- fabricante: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- clasificacion: string (nullable = true)
 |-- n_establecimientos: double (nullable = true)
 |-- precio_media: double (nullable = true)
 |-- precio_mediana: double (nullable = true)
 |-- precio_moda: double (nullable = true)
 |-- periodo: date (nullable = true)
 |-- mes: integer (nullable = true)
 |-- nombre_mes: string (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- anio: integer (null

In [0]:
# 3. Conversión de cada CSV a Parquet y tamaño en disco
from pyspark.sql import functions as F

def tam_mb(ruta):
    """Suma el tamaño de un archivo o de todos los archivos de una carpeta (Parquet), en MB."""
    archivos = dbutils.fs.ls(ruta)
    return round(sum(f.size for f in archivos) / 1024 / 1024, 2)

tamanos = []
for etiqueta, nombre in CASOS.items():
    csv_ruta = BASE + nombre + ".csv"
    pq_ruta = BASE + nombre + "_parquet"
    (spark.read.csv(csv_ruta, header=True, inferSchema=True)
         .write.mode("overwrite").parquet(pq_ruta))
    csv_mb, pq_mb = tam_mb(csv_ruta), tam_mb(pq_ruta)
    tamanos.append((etiqueta, csv_mb, pq_mb, round(100 * (1 - pq_mb / csv_mb), 1)))

df_tamanos = spark.createDataFrame(tamanos, ["escala", "csv_mb", "parquet_mb", "reduccion_pct"])
display(df_tamanos)   

escala,csv_mb,parquet_mb,reduccion_pct
100k,22.88,2.68,88.3
500k,114.57,11.58,89.9
1m,229.22,21.79,90.5


In [0]:
# 4. Función de medición: 1 calentamiento descartado + 3 repeticiones, mediana
import time, statistics
from datetime import datetime

def medir(accion, repeticiones=REPETICIONES):
    accion()                                   
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        accion()
        tiempos.append(time.perf_counter() - t0)
    return tiempos

def consulta(d):
    return d.groupBy(COL_GRUPO_1, COL_GRUPO_2).agg(F.avg(COL_VALOR).alias("precio_promedio")).collect()

filas = {"100k": 100_000, "500k": 500_000, "1m": 1_000_000}

In [0]:
# 5. Ejecución del experimento
registros = []
for etiqueta, nombre in CASOS.items():
    csv_ruta = BASE + nombre + ".csv"
    pq_ruta = BASE + nombre + "_parquet"
    pruebas = {
        "CSV": lambda r=csv_ruta: consulta(spark.read.csv(r, header=True, inferSchema=True)),
        "Parquet": lambda r=pq_ruta: consulta(spark.read.parquet(r)),
    }
    for formato, accion in pruebas.items():
        tiempos = medir(accion)
        mediana = statistics.median(tiempos)
        print(f"{formato:8} {etiqueta:5} | tiempos (s): {[round(t, 3) for t in tiempos]} | mediana (s): {mediana:.3f}")
        for i, t in enumerate(tiempos, start=1):
            registros.append((EXPERIMENTO, INTEGRANTE, f"{formato}_{etiqueta}", formato, etiqueta,
                              filas[etiqueta], i, round(t, 4), datetime.now()))

CSV      100k  | tiempos (s): [1.422, 1.273, 1.2] | mediana (s): 1.273
Parquet  100k  | tiempos (s): [0.852, 0.879, 0.819] | mediana (s): 0.852
CSV      500k  | tiempos (s): [1.614, 1.569, 1.533] | mediana (s): 1.569
Parquet  500k  | tiempos (s): [0.812, 0.823, 0.797] | mediana (s): 0.812
CSV      1m    | tiempos (s): [1.978, 2.088, 2.365] | mediana (s): 2.088
Parquet  1m    | tiempos (s): [0.796, 0.874, 0.83] | mediana (s): 0.830


In [0]:
# 6. Tabla de resultados para la ficha (repeticiones, mediana y throughput)
df_res = spark.createDataFrame(
    registros,
    "experimento string, integrante string, caso string, formato string, escala string, "
    "filas long, repeticion int, tiempo_s double, fecha timestamp")

ficha = (df_res.groupBy("caso", "formato", "escala", "filas")
         .pivot("repeticion", [1, 2, 3]).agg(F.first("tiempo_s"))
         .withColumnRenamed("1", "rep_1").withColumnRenamed("2", "rep_2").withColumnRenamed("3", "rep_3")
         .join(df_res.groupBy("caso").agg(F.percentile_approx("tiempo_s", 0.5).alias("mediana_s")), "caso")
         .withColumn("filas_por_s", F.round(F.col("filas") / F.col("mediana_s"), 0))
         .join(df_tamanos.select("escala", "csv_mb", "parquet_mb"), "escala")
         .orderBy("filas", "formato"))

display(ficha)   

escala,caso,formato,filas,rep_1,rep_2,rep_3,mediana_s,filas_por_s,csv_mb,parquet_mb
100k,CSV_100k,CSV,100000,1.4215,1.273,1.1995,1.273,78555.0,22.88,2.68
100k,Parquet_100k,Parquet,100000,0.852,0.8791,0.8193,0.852,117371.0,22.88,2.68
500k,CSV_500k,CSV,500000,1.6141,1.5687,1.5333,1.5687,318735.0,114.57,11.58
500k,Parquet_500k,Parquet,500000,0.8119,0.8225,0.7971,0.8119,615839.0,114.57,11.58
1m,CSV_1m,CSV,1000000,1.9782,2.088,2.3647,2.088,478927.0,229.22,21.79
1m,Parquet_1m,Parquet,1000000,0.7956,0.8745,0.8302,0.8302,1204529.0,229.22,21.79


In [0]:
# 7. Guardar en la tabla común de experimentos (valores medidos el 21/11/2026, celda 6)
from pyspark.sql import functions as F
from datetime import datetime

TABLA_RESULTADOS = "workspace.ef_precios_medicamentos.exp_resultados"
EXPERIMENTO, INTEGRANTE = "EXP_GONZALO_FORMATO", "Gonzalo"
FECHA = datetime(2026, 11, 21, 15, 0)   # hora aproximada de la medición

medidos = {
    ("CSV", "100k", 100000):      [1.4215, 1.2730, 1.1995],
    ("Parquet", "100k", 100000):  [0.8520, 0.8791, 0.8193],
    ("CSV", "500k", 500000):      [1.6141, 1.5687, 1.5333],
    ("Parquet", "500k", 500000):  [0.8119, 0.8225, 0.7971],
    ("CSV", "1m", 1000000):       [1.9782, 2.0880, 2.3647],
    ("Parquet", "1m", 1000000):   [0.7956, 0.8745, 0.8302],
}
registros = [(EXPERIMENTO, INTEGRANTE, f"{fmt}_{esc}", fmt, esc, filas, i, t, FECHA)
             for (fmt, esc, filas), tiempos in medidos.items()
             for i, t in enumerate(tiempos, start=1)]

df_res = spark.createDataFrame(registros,
    "experimento string, integrante string, caso string, formato string, escala string, "
    "filas long, repeticion int, tiempo_s double, fecha timestamp")

df_res.write.mode("append").option("mergeSchema", "true").saveAsTable(TABLA_RESULTADOS)
display(spark.table(TABLA_RESULTADOS).filter(F.col("experimento") == EXPERIMENTO).orderBy("filas", "formato", "repeticion"))

experimento,integrante,caso,formato,escala,filas,repeticion,tiempo_s,fecha
EXP_GONZALO_FORMATO,Gonzalo,CSV_100k,CSV,100k,100000,1,1.4215,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,CSV_100k,CSV,100k,100000,2,1.273,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,CSV_100k,CSV,100k,100000,3,1.1995,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,Parquet_100k,Parquet,100k,100000,1,0.852,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,Parquet_100k,Parquet,100k,100000,2,0.8791,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,Parquet_100k,Parquet,100k,100000,3,0.8193,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,CSV_500k,CSV,500k,500000,1,1.6141,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,CSV_500k,CSV,500k,500000,2,1.5687,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,CSV_500k,CSV,500k,500000,3,1.5333,2026-11-21T15:00:00.000Z
EXP_GONZALO_FORMATO,Gonzalo,Parquet_500k,Parquet,500k,500000,1,0.8119,2026-11-21T15:00:00.000Z


In [0]:
%sql
SELECT experimento, COUNT(*) AS filas
FROM workspace.ef_precios_medicamentos.exp_resultados
GROUP BY experimento;

experimento,filas
EXP_GONZALO_FORMATO,18
